In [ ]:
# ---- Versor bench: step time and evaluation time of the Versor vector field on one GPU ----------------------
# Runs pose3d/scripts/bench_step.py (synthetic images, no dataset) for each variant below, one after the other, and
# prints one JSON line per measurement: the training step (trainer's train_epoch, EMA on, batch 64), the 1-sample and
# 32-sample evaluation of 360 images. With ~1,220 steps per epoch of the reference recipe (USAGE.md: 215 s at
# 176 ms) this gives the ETA of a full Pascal3D+ run. No W&B, no data, no checkpoint.
#
# OFFLINE (RTX Pro 6000 kernels have no internet). Attach:
#   a dataset holding ga-research-<commit>.tar.gz (pose3d/scripts/make_snapshot.py) and, for the Python 3.13 image,
#     the healpy cp313 wheel (any *.whl next to it is installed too)
#   syfry5suvzovvakmuj/ga-research-offline-deps   (clifford, e3nn, healpy cp312, image2sphere wheels; ResNet-101 weights)
#
# WATCHDOG: every variant shares one deadline, MAX_MINUTES after the run cell starts; whatever runs then is killed, so
# the kernel stays under the 30-minute limit of CLAUDE.md without a human's approval (setup takes ~2 min).
MAX_MINUTES = 22
VARIANTS = {
    'versor_dense_compiled': ['--vector_field', 'versor'],
    'versor_matrix_compiled': ['--vector_field', 'versor', '--versor_linear', 'matrix'],
    'versor_dense_eager': ['--vector_field', 'versor', '--no-versor_compile'],
}
BENCH = ['--steps', '50', '--warmup', '5', '--eval_images', '360']
SNAPSHOT_MARKER = 'versor_compile'   # the snapshot must carry the Versor field with its speed options

In [ ]:
import glob, os, shutil, subprocess, sys, tarfile
if os.path.exists('/kaggle/working/GA-Research'):
    shutil.rmtree('/kaggle/working/GA-Research')
# Kaggle auto-extracts an uploaded tarball into a folder; accept that folder or the tarball itself.
markers = glob.glob('/kaggle/input/**/GA-Research/pose3d/pose3d/config.py', recursive=True)
tgzs = glob.glob('/kaggle/input/**/ga-research-*.tar.gz', recursive=True)
if markers:
    assert len(markers) == 1, f'attach exactly one repo snapshot, found {markers}'
    shutil.copytree(markers[0][:-len('/pose3d/pose3d/config.py')], '/kaggle/working/GA-Research')
else:
    assert len(tgzs) == 1, f'attach exactly one repo snapshot (ga-research-*.tar.gz), found {tgzs}'
    with tarfile.open(tgzs[0]) as t:
        t.extractall('/kaggle/working')
print(open('/kaggle/working/GA-Research/SNAPSHOT_COMMIT').read().strip())
assert SNAPSHOT_MARKER in open('/kaggle/working/GA-Research/pose3d/pose3d/config.py').read(), 'stale snapshot'
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip(), '| python', sys.version.split()[0])

In [ ]:
# Wheels from every attached dataset, only those this Python can run (pip rejects a whole command on one bad
# wheel), installed without dependencies so the image's torch stays. ResNet weights go to the torch hub cache.
import importlib
from packaging.tags import sys_tags
from packaging.utils import parse_wheel_filename
supported = set(sys_tags())
wheels = sorted(w for w in glob.glob('/kaggle/input/**/*.whl', recursive=True)
                if parse_wheel_filename(os.path.basename(w))[3] & supported)
print('wheels:', [os.path.basename(w) for w in wheels])
subprocess.run(['pip', 'install', '-q', '--no-index', '--no-deps', '--disable-pip-version-check', *wheels], check=True)
ckpt_dir = os.path.expanduser('~/.cache/torch/hub/checkpoints'); os.makedirs(ckpt_dir, exist_ok=True)
for w in glob.glob('/kaggle/input/**/resnet*-*.pth', recursive=True):
    shutil.copy(w, os.path.join(ckpt_dir, os.path.basename(w)))
for mod in ['e3nn', 'healpy', 'image2sphere', 'clifford']:
    importlib.import_module(mod)
print('deps import cleanly')

In [ ]:
# Each variant in its own process; one shared deadline. Output is also written to /kaggle/working/versor_bench.log,
# which Kaggle keeps as the kernel's output even when its log stream comes back empty.
import threading, time
os.chdir('/kaggle/working/GA-Research/pose3d')
deadline = time.time() + MAX_MINUTES * 60
with open('/kaggle/working/versor_bench.log', 'w') as fh:
    for name, flags in VARIANTS.items():
        left = deadline - time.time()
        if left < 120:
            print(f'SKIP {name}: {left:.0f} s left before the watchdog', flush=True); fh.write(f'SKIP {name}\n')
            continue
        cmd = [sys.executable, '-u', 'scripts/bench_step.py', *BENCH, '--', *flags]
        print(f'=== {name}:', ' '.join(cmd), flush=True); fh.write(f'=== {name}\n'); fh.flush()
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        timer = threading.Timer(left, lambda p=proc: (print('WATCHDOG: killing', flush=True), p.kill()))
        timer.start()
        for line in proc.stdout:
            print(line, end='', flush=True); fh.write(line); fh.flush()
        rc = proc.wait(); timer.cancel()
        print(f'=== {name}: exit code {rc}', flush=True); fh.write(f'=== {name}: exit code {rc}\n'); fh.flush()